In [2]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import optuna
import pacmap
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from pyDRMetrics.pyDRMetrics import DRMetrics

from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    adjusted_rand_score
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# Diretórios
# ============================================================

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

OPTUNA_DIR = RESULTS_DIR / "optuna"
EMBEDDINGS_DIR = RESULTS_DIR / "embeddings"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

for directory in [
    OPTUNA_DIR,
    EMBEDDINGS_DIR,
    METRICS_DIR,
    FIGURES_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# Carregamento da base
# ============================================================

df = pd.read_csv(
    DATA_DIR / "mad_datasets" / "VST_MAD_3000.csv",
    index_col=0
)

clinical_final = pd.read_csv(
    DATA_DIR / "clinical_final.csv",
    index_col=0
)

# Mantém apenas a coluna de subtipo
clinical_final = clinical_final[["SUBTYPE"]]

# Alinha a tabela clínica à matriz de expressão
clinical_final = clinical_final.loc[df.index]

# Verificações
assert df.index.equals(clinical_final.index), (
    "Os índices da matriz de expressão e da tabela clínica não coincidem."
)

assert not df.isnull().values.any(), (
    "A matriz de expressão contém valores ausentes."
)

assert not clinical_final["SUBTYPE"].isnull().any(), (
    "A tabela clínica contém subtipos ausentes."
)

X_data = df.values
y_true = clinical_final["SUBTYPE"].values

print(f"Número de amostras: {X_data.shape[0]}")
print(f"Número de genes: {X_data.shape[1]}")
print(f"Número de subtipos clínicos: {clinical_final['SUBTYPE'].nunique()}")

print("\nDistribuição dos subtipos:")
print(clinical_final["SUBTYPE"].value_counts())

c:\Users\willi\meu_projeto_python\Dissertação\RBras\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Número de amostras: 970
Número de genes: 3000
Número de subtipos clínicos: 5

Distribuição dos subtipos:
SUBTYPE
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64


## Função Objetivo Optuna

In [3]:
def executar_pacmap(X, params):
    """
    Executa o PaCMAP utilizando os hiperparâmetros informados.
    """

    reducer = pacmap.PaCMAP(
        n_components=params["n_components"],
        n_neighbors=params["n_neighbors"],
        MN_ratio=params["MN_ratio"],
        FP_ratio=params["FP_ratio"],
        random_state=42
    )

    Z = reducer.fit_transform(X, init="pca")

    return Z


def calcular_metricas_reducao(X, Z):
    """
    Calcula as métricas de preservação da estrutura dos dados.
    """

    dr_metrics = DRMetrics(X, Z)

    q_local = dr_metrics.Qlocal
    auc_qnn = dr_metrics.AUC

    return q_local, auc_qnn


def calcular_metricas_clustering(Z, labels):
    """
    Calcula Silhouette e Calinski-Harabasz.
    """

    n_clusters = len(np.unique(labels))

    if n_clusters < 2 or n_clusters >= len(labels):
        raise ValueError("Número inválido de clusters para avaliação.")

    sil_score = silhouette_score(Z, labels)

    ch_score = calinski_harabasz_score(Z, labels)

    # Mantém a normalização utilizada no código original
    ch_score_norm = ch_score / (ch_score + 1000)

    return sil_score, ch_score, ch_score_norm


def calcular_score_reducao(X, Z):
    """
    Score utilizado para avaliar a redução de dimensionalidade.
    """

    q_local, auc_qnn = calcular_metricas_reducao(X, Z)

    score = 0.5 * q_local + 0.5 * auc_qnn

    return score, q_local, auc_qnn


def calcular_score_clustering(Z, labels):
    """
    Score utilizado para avaliar o clustering.
    """

    sil_score, ch_score, ch_score_norm = (
        calcular_metricas_clustering(Z, labels)
    )

    score = 0.5 * sil_score + 0.5 * ch_score_norm

    return score, sil_score, ch_score, ch_score_norm

## Objetivo Optuna Conjunto

In [4]:
def objective_joint(trial):

    # ========================================================
    # Hiperparâmetros do PaCMAP
    # ========================================================

    pacmap_params = {
        "n_neighbors": trial.suggest_int(
            "n_neighbors", 10, 150
        ),

        "MN_ratio": trial.suggest_float(
            "MN_ratio", 0.1, 3.0
        ),

        "FP_ratio": trial.suggest_float(
            "FP_ratio", 0.1, 5.0
        ),

        "n_components": trial.suggest_int(
            "n_components", 2, 10
        )
    }

    # ========================================================
    # Redução de dimensionalidade
    # ========================================================

    Z = executar_pacmap(X_data, pacmap_params)

    try:
        score_dr, q_local, auc_qnn = calcular_score_reducao(
            X_data, Z
        )

    except Exception as error:
        print(f"Erro nas métricas de redução: {error}")
        raise optuna.TrialPruned()

    # ========================================================
    # Hiperparâmetros do GMM
    # ========================================================

    gmm_params = {
        "n_components": trial.suggest_int(
            "gmm_n_components", 2, 7
        ),

        "covariance_type": trial.suggest_categorical(
            "covariance_type",
            ["full", "tied", "diag", "spherical"]
        ),

        "n_init": trial.suggest_int(
            "n_init", 5, 10
        ),

        "random_state": 42
    }

    # ========================================================
    # Clustering
    # ========================================================

    try:
        gmm = GaussianMixture(**gmm_params)

        labels = gmm.fit_predict(Z)

        score_gmm, sil_score, ch_score, ch_score_norm = (
            calcular_score_clustering(Z, labels)
        )

    except Exception as error:
        print(f"Erro no GMM ou nas métricas: {error}")
        raise optuna.TrialPruned()

    # ========================================================
    # Score final
    # ========================================================

    score_final = 0.4 * score_dr + 0.6 * score_gmm

    # Armazena métricas adicionais no ensaio
    trial.set_user_attr("Qlocal", float(q_local))
    trial.set_user_attr("AUC_QNN", float(auc_qnn))
    trial.set_user_attr("Silhouette", float(sil_score))
    trial.set_user_attr("Calinski_Harabasz", float(ch_score))
    trial.set_user_attr("CH_normalizado", float(ch_score_norm))
    trial.set_user_attr("Score_DR", float(score_dr))
    trial.set_user_attr("Score_GMM", float(score_gmm))

    return score_final

## Execução do Optuna

In [5]:
study_joint = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="Joint_VST_MAD_3000",
    storage=f"sqlite:///{OPTUNA_DIR / 'joint_vst_mad_3000.db'}",
    load_if_exists=True
)

study_joint.optimize(
    objective_joint,
    n_trials=200
)

print("Melhor score:", study_joint.best_value)

print("\nMelhores hiperparâmetros:")
print(study_joint.best_params)

Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Sample size is smaller than the total number of assigned points, n_neighbors, n_MN, n_FP would be reorganized
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note:

Melhor score: 0.7733203705383359

Melhores hiperparâmetros:
{'n_neighbors': 94, 'MN_ratio': 0.21907209097316543, 'FP_ratio': 1.4578635291202184, 'n_components': 7, 'gmm_n_components': 2, 'covariance_type': 'diag', 'n_init': 9}


## Melhores parâmetros do Optuna Conjunto

In [6]:
best_params_joint = study_joint.best_params

# Parâmetros do PaCMAP
best_pacmap_params_joint = {
    "n_neighbors": best_params_joint["n_neighbors"],
    "MN_ratio": best_params_joint["MN_ratio"],
    "FP_ratio": best_params_joint["FP_ratio"],
    "n_components": best_params_joint["n_components"]
}

# Parâmetros do GMM
best_gmm_params_joint = {
    "n_components": best_params_joint["gmm_n_components"],
    "covariance_type": best_params_joint["covariance_type"],
    "n_init": best_params_joint["n_init"],
    "random_state": 42
}

# PaCMAP
Z_joint = executar_pacmap(
    X_data,
    best_pacmap_params_joint
)

# GMM
gmm_joint = GaussianMixture(
    **best_gmm_params_joint
)

labels_joint = gmm_joint.fit_predict(Z_joint)

# Métricas
score_dr_joint, q_local_joint, auc_qnn_joint = (
    calcular_score_reducao(X_data, Z_joint)
)

score_gmm_joint, sil_joint, ch_joint, ch_norm_joint = (
    calcular_score_clustering(Z_joint, labels_joint)
)

ari_joint = adjusted_rand_score(
    y_true,
    labels_joint
)

print("Resultados Joint")
print("------------------------------")
print(f"Qlocal: {q_local_joint:.4f}")
print(f"AUC QNN: {auc_qnn_joint:.4f}")
print(f"Silhouette: {sil_joint:.4f}")
print(f"Calinski-Harabasz: {ch_joint:.4f}")
print(f"ARI: {ari_joint:.4f}")

Note: `n_components != 2` have not been thoroughly tested.


Resultados Joint
------------------------------
Qlocal: 0.4134
AUC QNN: 0.7537
Silhouette: 0.8725
Calinski-Harabasz: 12733.0132
ARI: 0.3388


## Salvando os resultados

In [7]:
# Embedding
df_embedding_joint = pd.DataFrame(
    Z_joint,
    index=df.index,
    columns=[
        f"PaCMAP_{i+1}"
        for i in range(Z_joint.shape[1])
    ]
)

df_embedding_joint["Cluster"] = labels_joint
df_embedding_joint["SUBTYPE"] = y_true

df_embedding_joint.to_csv(
    EMBEDDINGS_DIR / "embedding_joint_vst_mad_3000.csv"
)

# Modelo GMM
joblib.dump(
    gmm_joint,
    EMBEDDINGS_DIR / "gmm_joint_vst_mad_3000.pkl"
)

# Hiperparâmetros
with open(
    OPTUNA_DIR / "best_params_joint_vst_mad_3000.json",
    "w"
) as file:
    json.dump(best_params_joint, file, indent=4)

# Histórico dos ensaios
study_joint.trials_dataframe().to_csv(
    OPTUNA_DIR / "trials_joint_vst_mad_3000.csv",
    index=False
)

## Função Objetivo PaCMAP separado

In [8]:
def objective_pacmap(trial):

    params = {
        "n_neighbors": trial.suggest_int(
            "n_neighbors", 10, 150
        ),

        "MN_ratio": trial.suggest_float(
            "MN_ratio", 0.1, 3.0
        ),

        "FP_ratio": trial.suggest_float(
            "FP_ratio", 0.1, 5.0
        ),

        "n_components": trial.suggest_int(
            "n_components", 2, 10
        )
    }

    Z = executar_pacmap(X_data, params)

    try:
        score_dr, q_local, auc_qnn = calcular_score_reducao(
            X_data, Z
        )

    except Exception as error:
        print(f"Erro nas métricas de redução: {error}")
        raise optuna.TrialPruned()

    trial.set_user_attr("Qlocal", float(q_local))
    trial.set_user_attr("AUC_QNN", float(auc_qnn))

    return score_dr

## Optuna PaCMAP separado

In [9]:
study_pacmap = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="PaCMAP_VST_MAD_3000",
    storage=f"sqlite:///{OPTUNA_DIR / 'pacmap_vst_mad_3000.db'}",
    load_if_exists=True
)

study_pacmap.optimize(
    objective_pacmap,
    n_trials=200
)

print("Melhor score:", study_pacmap.best_value)

print("\nMelhores hiperparâmetros:")
print(study_pacmap.best_params)

Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Sample size is smaller than the total number of assigned points, n_neighbors, n_MN, n_FP would be reorganized
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note: `n_components != 2` have not been thoroughly tested.
Note:

Melhor score: 0.5963743105668685

Melhores hiperparâmetros:
{'n_neighbors': 34, 'MN_ratio': 2.7897083635061724, 'FP_ratio': 3.8375462999711187, 'n_components': 9}


In [10]:
best_params_pacmap = study_pacmap.best_params

Z_decoupled = executar_pacmap(
    X_data,
    best_params_pacmap
)

score_dr_decoupled, q_local_decoupled, auc_qnn_decoupled = (
    calcular_score_reducao(X_data, Z_decoupled)
)

print(f"Qlocal: {q_local_decoupled:.4f}")
print(f"AUC QNN: {auc_qnn_decoupled:.4f}")
print(f"Score DR: {score_dr_decoupled:.4f}")

Note: `n_components != 2` have not been thoroughly tested.


Qlocal: 0.4420
AUC QNN: 0.7507
Score DR: 0.5964


## Função Objetivo GMM separado

In [11]:
def objective_gmm(trial):

    params = {
        "n_components": trial.suggest_int(
            "n_components", 2, 7
        ),

        "covariance_type": trial.suggest_categorical(
            "covariance_type",
            ["full", "tied", "diag", "spherical"]
        ),

        "n_init": trial.suggest_int(
            "n_init", 5, 10
        ),

        "random_state": 42
    }

    try:
        gmm = GaussianMixture(**params)

        labels = gmm.fit_predict(Z_decoupled)

        score_gmm, sil_score, ch_score, ch_score_norm = (
            calcular_score_clustering(Z_decoupled, labels)
        )

    except Exception as error:
        print(f"Erro no GMM ou nas métricas: {error}")
        raise optuna.TrialPruned()

    trial.set_user_attr("Silhouette", float(sil_score))
    trial.set_user_attr("Calinski_Harabasz", float(ch_score))
    trial.set_user_attr("CH_normalizado", float(ch_score_norm))

    return score_gmm

## Optuna GMM separado

In [12]:
study_gmm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="GMM_VST_MAD_3000",
    storage=f"sqlite:///{OPTUNA_DIR / 'gmm_vst_mad_3000.db'}",
    load_if_exists=True
)

study_gmm.optimize(
    objective_gmm,
    n_trials=200
)

print("Melhor score:", study_gmm.best_value)

print("\nMelhores hiperparâmetros:")
print(study_gmm.best_params)

Melhor score: 0.7831071321210208

Melhores hiperparâmetros:
{'n_components': 2, 'covariance_type': 'full', 'n_init': 10}


In [13]:
best_params_gmm = study_gmm.best_params

gmm_decoupled = GaussianMixture(
    n_components=best_params_gmm["n_components"],
    covariance_type=best_params_gmm["covariance_type"],
    n_init=best_params_gmm["n_init"],
    random_state=42
)

labels_decoupled = gmm_decoupled.fit_predict(Z_decoupled)

score_gmm_decoupled, sil_decoupled, ch_decoupled, ch_norm_decoupled = (
    calcular_score_clustering(Z_decoupled, labels_decoupled)
)

ari_decoupled = adjusted_rand_score(
    y_true,
    labels_decoupled
)

print("Resultados duas etapas")
print("------------------------------")
print(f"Qlocal: {q_local_decoupled:.4f}")
print(f"AUC QNN: {auc_qnn_decoupled:.4f}")
print(f"Silhouette: {sil_decoupled:.4f}")
print(f"Calinski-Harabasz: {ch_decoupled:.4f}")
print(f"ARI: {ari_decoupled:.4f}")

Resultados duas etapas
------------------------------
Qlocal: 0.4420
AUC QNN: 0.7507
Silhouette: 0.7662
Calinski-Harabasz: 3999.6978
ARI: 0.3327


## Salvando os resultados

In [14]:
df_embedding_decoupled = pd.DataFrame(
    Z_decoupled,
    index=df.index,
    columns=[
        f"PaCMAP_{i+1}"
        for i in range(Z_decoupled.shape[1])
    ]
)

df_embedding_decoupled["Cluster"] = labels_decoupled
df_embedding_decoupled["SUBTYPE"] = y_true

df_embedding_decoupled.to_csv(
    EMBEDDINGS_DIR / "embedding_decoupled_vst_mad_3000.csv"
)

joblib.dump(
    gmm_decoupled,
    EMBEDDINGS_DIR / "gmm_decoupled_vst_mad_3000.pkl"
)

with open(
    OPTUNA_DIR / "best_params_gmm_vst_mad_3000.json",
    "w"
) as file:
    json.dump(best_params_gmm, file, indent=4)

study_gmm.trials_dataframe().to_csv(
    OPTUNA_DIR / "trials_gmm_vst_mad_3000.csv",
    index=False
)

In [15]:
# Salvar parâmetros do PaCMAP
with open(OPTUNA_DIR / "best_params_pacmap_vst_mad_3000.json", "w") as file:
    json.dump(best_params_pacmap, file, indent=4)

# Salvar parâmetros do GMM
with open(OPTUNA_DIR / "best_params_gmm_vst_mad_3000.json", "w") as file:
    json.dump(best_params_gmm, file, indent=4)